In [1]:
import json
import re
import random
from pathlib import Path

In [2]:

def process_responses():
    resp_folder = Path('model_outputs')
    json_files = list(resp_folder.rglob('*.json'))
    
    visual_pattern = re.compile(r'8\|[\s\S]*?(?:F G H|FGH|. .)')
    all_sentences = []

    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                entries = json.load(f)
            except Exception:
                continue
            if not isinstance(entries, list):
                continue  # Skip files that are not lists

        for entry in entries:
            resp = entry.get('model_response', '')
            if not resp:
                continue
            cleaned = visual_pattern.sub('', resp)
            sentences = re.split(r'(?<=[.!?])\s+', cleaned.strip())
            # Only keep sentences within 20-200 chars
            all_sentences.extend(
                s for s in (s.strip() for s in sentences)
                if 20 <= len(s) <= 200
            )

    sample_count = min(100_000, len(all_sentences))
    sampled = random.sample(all_sentences, sample_count)

    out_folder = Path('processed_data')
    out_folder.mkdir(exist_ok=True)
    out_file = out_folder / f'sampled_sentences_{sample_count}.jsonl'
    with out_file.open('w', encoding='utf-8') as f:
        for sentence in sampled:
            json.dump({'sentence': sentence}, f, ensure_ascii=False)
            f.write('\n')

    print(f"Wrote {sample_count} sentences to {out_file.resolve()}")

In [3]:
process_responses()

Wrote 100000 sentences to C:\Users\lucas\Desktop\UCSD\Research\SP25 - Chess Reasoner\LLM_Chess\llm_chess\data\raw\processed_data\sampled_sentences_100000.jsonl
